In [22]:
import os
import requests
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
load_dotenv(override=True)
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = ChatGroq(model="openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
print(response.content, end="\n", flush=True)


Parrots don’t “talk” in the same way humans do—they’re exceptional mimics. Their ability to imitate human speech (and other sounds) comes from a combination of anatomy, brain structure, and social instincts.

### 1. Anatomy: a flexible vocal organ
- **Syrinx** – Birds produce sound with an organ called the syrinx, located at the base of the trachea. In parrots the syrinx is especially well‑developed and highly muscular, giving them fine control over pitch, tone, and rhythm.
- **Tongue & beak** – Their tongues are agile and their beaks can shape sounds, letting them reproduce a wide range of noises, from whistles to human consonants.

### 2. Brain wiring: vocal learning
- **Song‑learning circuitry** – Parrots belong to the group of “vocal learners,” like songbirds, hummingbirds, and humans. They have a specialized neural pathway (the “song system”) that lets them hear a sound, form a memory of it, and then reproduce it.
- **Highly social brain** – The same brain regions that process soc

In [23]:
from langchain.tools import tool
@tool
def get_weather(location: str)->str:
    """Get the weather at a location"""

    params = {
        "q": location,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric",   # °C; use "imperial" for °F
    }
    url = "https://api.openweathermap.org/data/2.5/weather"
    try:
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        if r.status_code != 200:
            return f"Could not get weather for {location}: {data.get('message', r.text)}"

        return (
            f"Weather in {data['name']}, {data['sys']['country']}: "
            f"{data['weather'][0]['description']}, "
            f"{data['main']['temp']}°C (feels like {data['main']['feels_like']}°C), "
            f"humidity {data['main']['humidity']}%, "
            f"wind {data['wind']['speed']} m/s."
        )
    except requests.RequestException as e:
        return f"Weather service error: {e}"

In [24]:
# agent = create_agent(model=model, tools=[get_weather], system_prompt="You are a helpful assistant.")

# result = agent.invoke(
#     {"messages": [{"role": "user", "content": "How's the weather in Hyderabad right now?"}]}
# )
# print(result["messages"][-1].content)
model_with_tools=model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call["name"]}")
    print(f"Args: {tool_call["args"]}")

Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [ ]:
messages = [{"role": "user","content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_result = model_with_tools.invoke(messages)
print(final_result.text)